# Run hierarchical PRA-AOG v6

This notebook assumes the v6 overlay is already present in the checkout. Edit paths in the first cell, then run top to bottom.


In [ ]:
!python -c 'from pathlib import Path; p=Path("/tmp/hier_pra_aog_v6.env"); p.write_text("export WORKSPACE=\"/home/dfli/instance_slot_aog\"\nexport REPO_DIR=\"$WORKSPACE/clean_v18_v39_v42\"\nexport PARTIMAGENET_ROOT=\"$WORKSPACE/full_hyco/PartImageNet\"\nexport STAGE1_CKPT=\"$REPO_DIR/runs/stage1_quality_upgrade/checkpoints/stage1_best.pt\"\nexport CONFIG=\"$REPO_DIR/configs/notebook_stage1.yaml\"\nexport CACHE_DIR=\"$REPO_DIR/artifacts/strict_aog_v6\"\nexport TRAIN_CACHE=\"$CACHE_DIR/train_strict_aog_terminals.pt\"\nexport VAL_CACHE=\"$CACHE_DIR/val_strict_aog_terminals.pt\"\nexport BUNDLE=\"$REPO_DIR/artifacts/pra_aog_v6/hier_pra_aog_v6_bundle.pt\"\nexport PART_TEMPLATE_BANK=\"$REPO_DIR/artifacts/pra_aog_v6/part_template_bank.pt\"\nexport RUN_DIR=\"$REPO_DIR/runs/hier_pra_aog_v6\"\nexport BEST_CKPT=\"$RUN_DIR/checkpoints/strict_aog_best.pt\"\n", encoding="utf-8"); print(p.read_text())'

## 1. Checkout, install, and smoke tests


In [ ]:
! . /tmp/hier_pra_aog_v6.env && git -C "$REPO_DIR" fetch origin pra-aog-v6-gpu-terminal-cache && git -C "$REPO_DIR" switch pra-aog-v6-gpu-terminal-cache && git -C "$REPO_DIR" pull --ff-only origin pra-aog-v6-gpu-terminal-cache && cd "$REPO_DIR" && python -m pip install -e ".[dev,vision]"

In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && pytest -q tests/test_pra_aog_v6_template_hierarchy.py tests/test_hier_pra_aog.py tests/test_strict_aog_core.py

## 2. Dataset config and terminal caches
Skip cache generation if `TRAIN_CACHE` and `VAL_CACHE` already exist.


In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && python -c 'from pathlib import Path; import os; p=Path(os.environ["CONFIG"]); p.write_text("extends: stage1_quality_upgrade.yaml\npaths:\n  partimagenet_root: " + repr(os.environ["PARTIMAGENET_ROOT"]) + "\n", encoding="utf-8"); print(p.read_text())'

In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && if [ -f "$TRAIN_CACHE" ] && [ -f "$VAL_CACHE" ]; then echo "Terminal caches already exist."; else mkdir -p "$CACHE_DIR"; python scripts/cache_strict_aog_terminals.py --config "$CONFIG" --stage1-ckpt "$STAGE1_CKPT" --out-dir "$CACHE_DIR" --device auto --splits train,val --batch-size 16 --num-workers 2 --threshold 0.40 --support-gate-mode post --support-component-mode best --max-terminals 32 --mask-size 64 --shard-size 1024 --store-images --store-images-splits val; fi

## 3. Build v6 bundle and part-template bank


In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && mkdir -p "$(dirname "$BUNDLE")" && python scripts/build_hier_pra_aog_v6.py --cache "$TRAIN_CACHE" --out "$BUNDLE" --part-template-out "$PART_TEMPLATE_BANK" --part-template-grid-size 3 --part-template-min-cell-coverage 0.06 --part-template-min-support 6 --part-template-min-cells 2 --part-template-max-per-part 6 --part-template-max-subparts 6 --part-template-mdl-cell-penalty 0.03 --part-template-score-boost 0.30

## 4. Smoke train, full train, evaluate


In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog_v6.py --bundle "$BUNDLE" --part-template-bank "$PART_TEMPLATE_BANK" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "${RUN_DIR}_smoke" --device auto --batch-size 4 --epochs 1 --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.30 --part-template-score-weight 0.30 --max-train-batches 2 --max-val-batches 2 --num-workers 0

In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog_v6.py --bundle "$BUNDLE" --part-template-bank "$PART_TEMPLATE_BANK" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$RUN_DIR" --device auto --batch-size 16 --epochs 20 --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.30 --part-template-score-weight 0.30 --preload-cache --num-workers 0

In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog_v6.py --bundle "$BUNDLE" --part-template-bank "$PART_TEMPLATE_BANK" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$RUN_DIR" --checkpoint "$BEST_CKPT" --eval-only --device auto --batch-size 16 --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --posterior-logits --preload-cache --num-workers 0

## 5. Decode one validation image


In [ ]:
! . /tmp/hier_pra_aog_v6.env && cd "$REPO_DIR" && python scripts/infer_hier_pra_aog_v6.py --bundle "$BUNDLE" --part-template-bank "$PART_TEMPLATE_BANK" --cache "$VAL_CACHE" --checkpoint "$BEST_CKPT" --out-dir "$RUN_DIR/inference" --sample-index 0 --device auto --assignment gpu_mf --top-k 5 --posterior-tau 0.75